# 02. TensorFlow / Keras ANN Model for PMSM Fault Classification

This notebook demonstrates training and evaluating an Artificial Neural Network (ANN) built with TensorFlow/Keras to detect and classify PMSM motor faults.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

# Add project root to sys.path
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from src.data_loader import load_and_preprocess_data
from src.models import build_tf_ann_model

## 1. Data Ingestion & Preprocessing

In [ ]:
data_path = repo_root / 'data' / 'Classification.csv'
data = load_and_preprocess_data(data_path=data_path)
print(f"Training set shape: {data['X_train'].shape}")
print(f"Validation set shape: {data['X_val'].shape}")
print(f"Test set shape: {data['X_test'].shape}")
print(f"Fault Target Classes: {data['classes'].tolist()}")

## 2. Build & Compile Keras ANN Model

In [ ]:
input_dim = data['X_train'].shape[1]
num_classes = len(data['classes'])
model = build_tf_ann_model(input_dim=input_dim, num_classes=num_classes, hidden_units=(128, 64))
model.summary()

## 3. Train Model with Callback Monitoring

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
history = model.fit(
    data['X_train'], data['y_train'],
    epochs=35,
    batch_size=64,
    validation_data=(data['X_val'], data['y_val']),
    callbacks=[early_stopping],
    verbose=1
)

## 4. Evaluate Test Set Performance & Confusion Matrix

In [ ]:
test_loss, test_acc = model.evaluate(data['X_test'], data['y_test'], verbose=0)
print(f"Test Accuracy: {test_acc * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

y_pred_probs = model.predict(data['X_test'])
y_pred = np.argmax(y_pred_probs, axis=1)

print("\nClassification Report:")
print(classification_report(data['y_test'], y_pred, target_names=[str(c) for c in data['classes']]))

# Plot Confusion Matrix
cm = confusion_matrix(data['y_test'], y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=data['classes'], yticklabels=data['classes'])
plt.title('PMSM Fault Classification Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()